# Trade Qualityの既存実験

**Historical archive / 過去の研究記録**

原本のコードを保持しています。独立実行や現在の検証基準への適合は保証しません。前のセルの変数に依存する箇所があります。実行入口は `../08_trade_quality.ipynb` を参照してください。

保存出力は `../../results/legacy/`、既知の問題は `../../docs/AUDIT.md` に整理しています。


## 元Notebookのセル 16

出典: `FX.ipynb`、0始まりのindex=15。コード内容は変更していません。

In [ ]:
# ============================================================
# USD/JPY 5分足
# Trade Quality AI
#
# MOVE AI
#     ↓
# Direction AI
#     ↓
# Quality AI
#     ↓
# BASE vs QUALITY FILTER
#
# 完全Walk-Forward比較
# ============================================================


# ============================================================
# 1. ライブラリ
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier


# ============================================================
# 2. 基本設定
# ============================================================

# 30分 = 5分足6本
HORIZON_BARS = 6


# 30分で±0.05%以上動けばMOVE
MOVE_THRESHOLD = 0.0005


# 仮の取引コスト
TRADING_COST = 0.0000133


# 過去データの時間減衰
HALF_LIFE_DAYS = 20


# Outer Walk-Forward
OUTER_SPLITS = 5


# Quality AI用の
# 内部Out-of-Fold分割
QUALITY_OOF_SPLITS = 4


# Validationで最低必要な取引数
MIN_VALIDATION_TRADES = 15


# ------------------------------------------------------------
# 閾値候補
# ------------------------------------------------------------

MOVE_PROB_LIST = [
    0.55,
    0.60,
    0.65,
    0.70
]


DIRECTION_PROB_LIST = [
    0.55,
    0.60,
    0.65,
    0.70
]


QUALITY_PROB_LIST = [
    0.50,
    0.55,
    0.60,
    0.65,
    0.70
]


# ------------------------------------------------------------
# TP / SL候補
# ------------------------------------------------------------

TP_LIST = [
    0.0005,   # 0.05%
    0.0008,   # 0.08%
    0.0010    # 0.10%
]


SL_LIST = [
    0.0005,   # 0.05%
    0.0007,   # 0.07%
    0.0010    # 0.10%
]


# ============================================================
# 3. 必要な特徴量を確認・作成
# ============================================================

# ------------------------------------------------------------
# リターン
# ------------------------------------------------------------

df["return_5m"] = (
    df["Close"]
    .pct_change(1)
)

df["return_15m"] = (
    df["Close"]
    .pct_change(3)
)

df["return_30m"] = (
    df["Close"]
    .pct_change(6)
)

df["return_1h"] = (
    df["Close"]
    .pct_change(12)
)

df["return_2h"] = (
    df["Close"]
    .pct_change(24)
)


# ============================================================
# 4. 移動平均
# ============================================================

df["MA5"] = (
    df["Close"]
    .rolling(5)
    .mean()
)

df["MA20"] = (
    df["Close"]
    .rolling(20)
    .mean()
)

df["MA50"] = (
    df["Close"]
    .rolling(50)
    .mean()
)


df["MA5_distance"] = (
    df["Close"]
    / df["MA5"]
    - 1
)

df["MA20_distance"] = (
    df["Close"]
    / df["MA20"]
    - 1
)

df["MA50_distance"] = (
    df["Close"]
    / df["MA50"]
    - 1
)


df["MA5_slope"] = (
    df["MA5"]
    .pct_change(3)
)

df["MA20_slope"] = (
    df["MA20"]
    .pct_change(3)
)

df["MA50_slope"] = (
    df["MA50"]
    .pct_change(3)
)


# ============================================================
# 5. ローソク足
# ============================================================

df["body"] = (

    abs(
        df["Close"]
        - df["Open"]
    )

    / df["Open"]
)


df["range"] = (

    df["High"]
    - df["Low"]

) / df["Close"]


df["upper_wick"] = (

    df["High"]

    - df[
        [
            "Open",
            "Close"
        ]
    ].max(
        axis=1
    )

) / df["Close"]


df["lower_wick"] = (

    df[
        [
            "Open",
            "Close"
        ]
    ].min(
        axis=1
    )

    - df["Low"]

) / df["Close"]


df["bullish"] = (

    df["Close"]
    > df["Open"]

).astype(int)


# ============================================================
# 6. ボラティリティ
# ============================================================

df["volatility_1h"] = (

    df["return_5m"]
    .rolling(12)
    .std()
)


df["volatility_2h"] = (

    df["return_5m"]
    .rolling(24)
    .std()
)


df["volatility_4h"] = (

    df["return_5m"]
    .rolling(48)
    .std()
)


# ============================================================
# 7. RSI
# ============================================================

delta = (
    df["Close"]
    .diff()
)


gain = (
    delta
    .clip(
        lower=0
    )
)


loss = (
    -delta
    .clip(
        upper=0
    )
)


avg_gain = (
    gain
    .rolling(14)
    .mean()
)


avg_loss = (
    loss
    .rolling(14)
    .mean()
)


rs = (
    avg_gain
    / avg_loss
)


df["RSI"] = (

    100

    - 100
    / (
        1 + rs
    )
)


# ============================================================
# 8. 高値・安値からの距離
# ============================================================

df["high_1h"] = (

    df["High"]
    .rolling(12)
    .max()
)


df["low_1h"] = (

    df["Low"]
    .rolling(12)
    .min()
)


df["distance_high_1h"] = (

    df["Close"]
    / df["high_1h"]
    - 1
)


df["distance_low_1h"] = (

    df["Close"]
    / df["low_1h"]
    - 1
)


# ============================================================
# 9. 時刻特徴
# ============================================================

df["hour"] = (
    df.index.hour
)


df["weekday"] = (
    df.index.dayofweek
)


df["hour_sin"] = np.sin(

    2
    * np.pi
    * df["hour"]
    / 24
)


df["hour_cos"] = np.cos(

    2
    * np.pi
    * df["hour"]
    / 24
)


# ============================================================
# 10. ATR
# ============================================================

previous_close = (
    df["Close"]
    .shift(1)
)


tr1 = (
    df["High"]
    - df["Low"]
)


tr2 = abs(

    df["High"]
    - previous_close
)


tr3 = abs(

    df["Low"]
    - previous_close
)


true_range = pd.concat(

    [
        tr1,
        tr2,
        tr3
    ],

    axis=1

).max(
    axis=1
)


df["ATR14"] = (

    true_range
    .rolling(14)
    .mean()
)


df["ATR14_pct"] = (

    df["ATR14"]
    / df["Close"]
)


# ============================================================
# 11. ADX
# ============================================================

high_diff = (
    df["High"]
    .diff()
)


low_diff = (
    -df["Low"]
    .diff()
)


plus_dm = np.where(

    (
        high_diff
        > low_diff
    )

    &

    (
        high_diff > 0
    ),

    high_diff,

    0.0
)


minus_dm = np.where(

    (
        low_diff
        > high_diff
    )

    &

    (
        low_diff > 0
    ),

    low_diff,

    0.0
)


plus_dm = pd.Series(

    plus_dm,

    index=df.index
)


minus_dm = pd.Series(

    minus_dm,

    index=df.index
)


atr_adx = (

    true_range
    .rolling(14)
    .mean()
)


plus_di = (

    100

    * plus_dm
    .rolling(14)
    .mean()

    / atr_adx
)


minus_di = (

    100

    * minus_dm
    .rolling(14)
    .mean()

    / atr_adx
)


dx = (

    100

    * abs(
        plus_di
        - minus_di
    )

    /

    (
        plus_di
        + minus_di
    )
)


df["ADX14"] = (

    dx
    .rolling(14)
    .mean()
)


# ============================================================
# 12. 時間定義を統一
# ============================================================

# シグナル時刻 = t
#
# 次足Open = t+1 でエントリー
#
# 30分なら
# t+1 ～ t+6
#
# t+6 Closeで終了

df["entry_price"] = (

    df["Open"]
    .shift(-1)
)


df["exit_price"] = (

    df["Close"]
    .shift(
        -HORIZON_BARS
    )
)


df["future_return"] = (

    df["exit_price"]

    / df["entry_price"]

    - 1
)


# ============================================================
# 13. 教師ラベル
# ============================================================

df["move_target"] = (

    abs(
        df["future_return"]
    )

    > MOVE_THRESHOLD

).astype(int)


df["direction_target"] = (

    df["future_return"]
    > 0

).astype(int)


# ============================================================
# 14. MOVE特徴量
# ============================================================

move_features = [

    "volatility_1h",
    "volatility_2h",
    "volatility_4h",

    "range",
    "body",

    "return_5m",
    "return_15m",
    "return_30m",

    "MA20_slope",
    "MA50_slope",

    "distance_high_1h",
    "distance_low_1h",

    "hour_sin",
    "hour_cos",

    "weekday"
]


# ============================================================
# 15. Direction特徴量
# ============================================================

direction_features = [

    "return_5m",
    "return_15m",
    "return_30m",
    "return_1h",
    "return_2h",

    "MA5_distance",
    "MA20_distance",
    "MA50_distance",

    "MA5_slope",
    "MA20_slope",
    "MA50_slope",

    "RSI",

    "bullish",

    "body",
    "upper_wick",
    "lower_wick",

    "distance_high_1h",
    "distance_low_1h",

    "volatility_1h",

    "hour_sin",
    "hour_cos",

    "weekday"
]


# ============================================================
# 16. Quality AIが見る市場特徴
# ============================================================

quality_market_features = [

    "return_5m",
    "return_15m",
    "return_30m",
    "return_1h",
    "return_2h",

    "volatility_1h",
    "volatility_2h",
    "volatility_4h",

    "ATR14_pct",
    "ADX14",

    "RSI",

    "MA5_distance",
    "MA20_distance",
    "MA50_distance",

    "MA5_slope",
    "MA20_slope",
    "MA50_slope",

    "distance_high_1h",
    "distance_low_1h",

    "body",
    "range",
    "upper_wick",
    "lower_wick",

    "hour_sin",
    "hour_cos",

    "weekday"
]


# ============================================================
# 17. DataFrame完成
# ============================================================

required_columns = list(

    set(

        move_features

        + direction_features

        + quality_market_features

        + [

            "future_return",

            "entry_price",
            "exit_price",

            "move_target",
            "direction_target"

        ]
    )
)


data = (

    df[
        required_columns
    ]

    .dropna()

    .copy()
)


print(
    "使用可能データ:",
    len(data)
)


# ============================================================
# 18. 時間減衰ウェイト
# ============================================================

def make_time_weights(
    index,
    half_life_days
):


    latest = (
        index.max()
    )


    age_days = (

        latest
        - index

    ).total_seconds() / 86400


    weights = (

        0.5

        ** (
            age_days
            / half_life_days
        )
    )


    return np.array(
        weights
    )


# ============================================================
# 19. Base AIを学習する関数
# ============================================================

def fit_base_models(
    train_frame,
    trees=250
):


    # --------------------------------------------------------
    # MOVE
    # --------------------------------------------------------

    move_weights = (
        make_time_weights(

            train_frame.index,

            HALF_LIFE_DAYS
        )
    )


    move_model = (

        RandomForestClassifier(

            n_estimators=
                trees,

            max_depth=8,

            min_samples_leaf=20,

            max_features="sqrt",

            class_weight="balanced",

            random_state=42,

            n_jobs=-1
        )
    )


    move_model.fit(

        train_frame[
            move_features
        ],

        train_frame[
            "move_target"
        ],

        sample_weight=
            move_weights
    )


    # --------------------------------------------------------
    # Direction
    # --------------------------------------------------------

    direction_train = (

        train_frame[

            train_frame[
                "move_target"
            ] == 1

        ]
    )


    if (

        len(
            direction_train
        ) < 50

        or

        direction_train[
            "direction_target"
        ].nunique() < 2

    ):

        return None


    direction_weights = (
        make_time_weights(

            direction_train.index,

            HALF_LIFE_DAYS
        )
    )


    direction_model = (

        RandomForestClassifier(

            n_estimators=
                trees,

            max_depth=8,

            min_samples_leaf=15,

            max_features="sqrt",

            class_weight="balanced",

            random_state=42,

            n_jobs=-1
        )
    )


    direction_model.fit(

        direction_train[
            direction_features
        ],

        direction_train[
            "direction_target"
        ],

        sample_weight=
            direction_weights
    )


    return (

        move_model,

        direction_model
    )


# ============================================================
# 20. Base AI確率予測
# ============================================================

def predict_base_models(
    models,
    frame
):


    move_model, \
    direction_model = models


    p_move = (

        move_model
        .predict_proba(

            frame[
                move_features
            ]

        )[:, 1]
    )


    direction_prob = (

        direction_model
        .predict_proba(

            frame[
                direction_features
            ]

        )
    )


    class_map = {

        c: i

        for i, c

        in enumerate(
            direction_model.classes_
        )
    }


    p_down = (

        direction_prob[
            :,
            class_map[0]
        ]
    )


    p_up = (

        direction_prob[
            :,
            class_map[1]
        ]
    )


    return (

        p_move,
        p_up,
        p_down
    )


# ============================================================
# 21. Quality特徴量作成
# ============================================================

def build_quality_features(

    frame,

    p_move,

    p_up,

    p_down

):


    quality_x = (

        frame[
            quality_market_features
        ]

        .copy()
    )


    # AI①の自信
    quality_x[
        "p_move"
    ] = p_move


    # AI②の確率
    quality_x[
        "p_up"
    ] = p_up


    quality_x[
        "p_down"
    ] = p_down


    # UP / DOWNどちらに
    # どれだけ自信があるか
    quality_x[
        "direction_confidence"
    ] = abs(

        p_up
        - p_down
    )


    # AIが選んだ方向
    quality_x[
        "predicted_direction"
    ] = (

        p_up
        >= p_down

    ).astype(int)


    return quality_x


# ============================================================
# 22. Quality教師ラベル
# ============================================================

def make_quality_target(

    frame,

    p_up,

    p_down

):


    # AIがBUYを選んだなら
    # future_returnそのまま

    predicted_buy = (

        p_up
        >= p_down

    )


    directional_return = np.where(

        predicted_buy,

        frame[
            "future_return"
        ].values,

        -frame[
            "future_return"
        ].values
    )


    # 取引コストを引く
    net_return = (

        directional_return

        - TRADING_COST
    )


    # プラスならQuality=1
    quality_target = (

        net_return > 0

    ).astype(int)


    return (

        quality_target,

        net_return
    )


# ============================================================
# 23. OOF Qualityデータ作成
# ============================================================

def create_quality_oof_dataset(
    frame
):


    # 最初40%は
    # 最初のBase学習に使う
    initial_size = int(

        len(frame)
        * 0.40
    )


    remaining = (

        len(frame)
        - initial_size
    )


    block = max(

        remaining

        // QUALITY_OOF_SPLITS,

        1
    )


    qx_list = []

    qy_list = []


    for split in range(
        QUALITY_OOF_SPLITS
    ):


        val_start = (

            initial_size

            + split
            * block
        )


        if split == QUALITY_OOF_SPLITS - 1:

            val_end = (
                len(frame)
            )

        else:

            val_end = min(

                val_start
                + block,

                len(frame)
            )


        # ----------------------------------------------------
        # GAP
        # ----------------------------------------------------

        train_end = (

            val_start
            - HORIZON_BARS
        )


        if train_end < 200:

            continue


        oof_train = (

            frame.iloc[
                :train_end
            ]
        )


        oof_val = (

            frame.iloc[
                val_start:val_end
            ]
        )


        if len(
            oof_val
        ) == 0:

            continue


        models = fit_base_models(

            oof_train,

            trees=180
        )


        if models is None:

            continue


        p_move, \
        p_up, \
        p_down = predict_base_models(

            models,

            oof_val
        )


        quality_x = (
            build_quality_features(

                oof_val,

                p_move,

                p_up,

                p_down
            )
        )


        quality_y, _ = (
            make_quality_target(

                oof_val,

                p_up,

                p_down
            )
        )


        quality_x[
            "quality_target"
        ] = quality_y


        qx_list.append(
            quality_x
        )


    if len(
        qx_list
    ) == 0:

        return None


    quality_data = (

        pd.concat(
            qx_list
        )

        .sort_index()
    )


    return quality_data


# ============================================================
# 24. Qualityモデル学習
# ============================================================

def fit_quality_model(
    quality_data
):


    quality_feature_names = [

        c

        for c in quality_data.columns

        if c != "quality_target"
    ]


    if (

        len(
            quality_data
        ) < 100

        or

        quality_data[
            "quality_target"
        ].nunique() < 2

    ):

        return None


    weights = (

        make_time_weights(

            quality_data.index,

            HALF_LIFE_DAYS
        )
    )


    model = (

        RandomForestClassifier(

            n_estimators=300,

            max_depth=7,

            min_samples_leaf=20,

            max_features="sqrt",

            class_weight="balanced",

            random_state=123,

            n_jobs=-1
        )
    )


    model.fit(

        quality_data[
            quality_feature_names
        ],

        quality_data[
            "quality_target"
        ],

        sample_weight=
            weights
    )


    return (

        model,

        quality_feature_names
    )


# ============================================================
# 25. Quality確率
# ============================================================

def predict_quality(

    quality_bundle,

    frame,

    p_move,

    p_up,

    p_down

):


    model, \
    feature_names = quality_bundle


    x = build_quality_features(

        frame,

        p_move,

        p_up,

        p_down
    )


    probability = (

        model
        .predict_proba(

            x[
                feature_names
            ]

        )[:, 1]
    )


    return probability


# ============================================================
# 26. TP / SL売買
# ============================================================

def simulate_trade(

    signal_time,

    direction,

    tp,

    sl

):


    try:

        signal_loc = (

            df.index
            .get_loc(
                signal_time
            )
        )

    except KeyError:

        return np.nan


    entry_loc = (

        signal_loc + 1
    )


    exit_loc = (

        signal_loc

        + HORIZON_BARS
    )


    if exit_loc >= len(df):

        return np.nan


    entry_price = (

        df.iloc[
            entry_loc
        ]["Open"]
    )


    # BUY
    if direction == "BUY":


        tp_price = (

            entry_price
            * (
                1 + tp
            )
        )


        sl_price = (

            entry_price
            * (
                1 - sl
            )
        )


    # SELL
    else:


        tp_price = (

            entry_price
            * (
                1 - tp
            )
        )


        sl_price = (

            entry_price
            * (
                1 + sl
            )
        )


    # --------------------------------------------------------
    # t+1 ～ t+6を見る
    # --------------------------------------------------------

    for loc in range(

        entry_loc,

        exit_loc + 1

    ):


        high = (

            df.iloc[
                loc
            ]["High"]
        )


        low = (

            df.iloc[
                loc
            ]["Low"]
        )


        if direction == "BUY":


            tp_hit = (

                high >= tp_price
            )


            sl_hit = (

                low <= sl_price
            )


        else:


            tp_hit = (

                low <= tp_price
            )


            sl_hit = (

                high >= sl_price
            )


        # 同じ5分足内なら
        # 保守的にSL先行
        if tp_hit and sl_hit:


            return (

                -sl

                - TRADING_COST
            )


        if tp_hit:


            return (

                tp

                - TRADING_COST
            )


        if sl_hit:


            return (

                -sl

                - TRADING_COST
            )


    # --------------------------------------------------------
    # TP/SL未到達
    # --------------------------------------------------------

    exit_price = (

        df.iloc[
            exit_loc
        ]["Close"]
    )


    if direction == "BUY":


        r = (

            exit_price
            / entry_price
            - 1
        )


    else:


        r = (

            entry_price
            / exit_price
            - 1
        )


    return (

        r

        - TRADING_COST
    )


# ============================================================
# 27. シグナル生成
# ============================================================

def make_signals(

    p_move,

    p_up,

    p_down,

    move_t,

    direction_t,

    quality_prob=None,

    quality_t=None

):


    buy = (

        (p_move >= move_t)

        &

        (p_up >= direction_t)

        &

        (p_up > p_down)
    )


    sell = (

        (p_move >= move_t)

        &

        (p_down >= direction_t)

        &

        (p_down > p_up)
    )


    # Quality Filter
    if (

        quality_prob is not None

        and

        quality_t is not None

    ):


        quality_mask = (

            quality_prob
            >= quality_t
        )


        buy = (

            buy

            & quality_mask
        )


        sell = (

            sell

            & quality_mask
        )


    signals = np.zeros(

        len(p_move)
    )


    signals[
        buy
    ] = 1


    signals[
        sell
    ] = -1


    return signals


# ============================================================
# 28. バックテスト
# ============================================================

def run_backtest(

    frame,

    signals,

    tp,

    sl

):


    returns = []

    times = []

    directions = []

    quality_indices = []


    i = 0


    while i < len(
        frame
    ):


        signal = (
            signals[i]
        )


        if signal == 0:


            i += 1

            continue


        if signal == 1:

            direction = "BUY"

        else:

            direction = "SELL"


        time = (
            frame.index[i]
        )


        r = simulate_trade(

            signal_time=
                time,

            direction=
                direction,

            tp=
                tp,

            sl=
                sl
        )


        if not np.isnan(r):


            returns.append(
                r
            )


            times.append(
                time
            )


            directions.append(
                direction
            )


            quality_indices.append(
                i
            )


        # 30分保有中は
        # 新規ポジションなし
        i += HORIZON_BARS


    return {

        "returns":
            np.array(
                returns
            ),

        "times":
            times,

        "directions":
            directions,

        "indices":
            quality_indices
    }


# ============================================================
# 29. Profit Factor
# ============================================================

def profit_factor(
    returns
):


    returns = np.array(
        returns
    )


    profits = (

        returns[
            returns > 0
        ].sum()
    )


    losses = abs(

        returns[
            returns < 0
        ].sum()
    )


    if losses == 0:

        return np.nan


    return (

        profits
        / losses
    )


# ============================================================
# 30. 統計関数
# ============================================================

def strategy_stats(
    returns
):


    returns = np.array(
        returns
    )


    if len(
        returns
    ) == 0:


        return {

            "trades":
                0,

            "win_rate":
                np.nan,

            "avg_return":
                np.nan,

            "profit_factor":
                np.nan,

            "max_dd":
                np.nan
        }


    equity = (

        1

        + pd.Series(
            returns
        )

    ).cumprod()


    running_max = (
        equity.cummax()
    )


    dd = (

        equity

        / running_max

        - 1
    )


    return {

        "trades":
            len(
                returns
            ),

        "win_rate":
            (
                returns > 0
            ).mean(),

        "avg_return":
            returns.mean(),

        "profit_factor":
            profit_factor(
                returns
            ),

        "max_dd":
            dd.min()
    }


# ============================================================
# 31. Outer Walk-Forward
# ============================================================

block_size = (

    len(data)

    // (
        OUTER_SPLITS + 1
    )
)


fold_results = []


all_base_returns = []

all_quality_returns = []


all_base_times = []

all_quality_times = []


# ============================================================
# 32. Foldループ
# ============================================================

for fold in range(
    OUTER_SPLITS
):


    print(
        "\n\n======================================"
    )


    print(
        "FOLD",
        fold + 1
    )


    print(
        "======================================"
    )


    # --------------------------------------------------------
    # Outer Train / Test
    # --------------------------------------------------------

    train_end = (

        block_size

        * (
            fold + 1
        )
    )


    test_start = (

        train_end

        + HORIZON_BARS
    )


    test_end = min(

        test_start
        + block_size,

        len(data)
    )


    train_full = (

        data.iloc[
            :train_end
        ]
    )


    test = (

        data.iloc[
            test_start:test_end
        ]
    )


    if (

        len(train_full) < 500

        or

        len(test) == 0

    ):


        print(
            "データ不足"
        )

        continue


    # --------------------------------------------------------
    # Validationを20%
    # --------------------------------------------------------

    split = int(

        len(train_full)
        * 0.80
    )


    train_core = (

        train_full.iloc[
            :split
            - HORIZON_BARS
        ]
    )


    validation = (

        train_full.iloc[
            split:
        ]
    )


    # ========================================================
    # 33. train_coreから
    # OOF Qualityデータを作る
    # ========================================================

    quality_oof = (

        create_quality_oof_dataset(
            train_core
        )
    )


    if quality_oof is None:


        print(
            "Quality OOF作成失敗"
        )

        continue


    print(
        "Quality学習データ:",
        len(
            quality_oof
        )
    )


    print(
        "Quality勝率:",
        round(

            quality_oof[
                "quality_target"
            ].mean()
            * 100,

            2
        ),
        "%"
    )


    quality_bundle = (

        fit_quality_model(
            quality_oof
        )
    )


    if quality_bundle is None:


        print(
            "Qualityモデル学習失敗"
        )

        continue


    # ========================================================
    # 34. Baseモデルをtrain_coreで学習
    # ========================================================

    base_models = (

        fit_base_models(

            train_core,

            trees=300
        )
    )


    if base_models is None:

        continue


    # ========================================================
    # 35. Validation予測
    # ========================================================

    val_p_move, \
    val_p_up, \
    val_p_down = (

        predict_base_models(

            base_models,

            validation
        )
    )


    val_quality_prob = (

        predict_quality(

            quality_bundle,

            validation,

            val_p_move,

            val_p_up,

            val_p_down
        )
    )


    # ========================================================
    # 36. BASE設定をValidationで決める
    # ========================================================

    best_base_score = -999

    best_base = None


    for move_t in MOVE_PROB_LIST:

        for direction_t in DIRECTION_PROB_LIST:

            for tp in TP_LIST:

                for sl in SL_LIST:


                    signals = make_signals(

                        val_p_move,

                        val_p_up,

                        val_p_down,

                        move_t,

                        direction_t
                    )


                    result = run_backtest(

                        validation,

                        signals,

                        tp,

                        sl
                    )


                    r = (
                        result[
                            "returns"
                        ]
                    )


                    if len(r) < MIN_VALIDATION_TRADES:

                        continue


                    # ----------------------------------------
                    # 期待値 × sqrt(取引数)
                    #
                    # 数件の偶然勝ちを
                    # 選びにくくする
                    # ----------------------------------------

                    score = (

                        r.mean()

                        * np.sqrt(
                            len(r)
                        )
                    )


                    if score > best_base_score:


                        best_base_score = (
                            score
                        )


                        best_base = {

                            "move_t":
                                move_t,

                            "direction_t":
                                direction_t,

                            "tp":
                                tp,

                            "sl":
                                sl
                        }


    if best_base is None:


        print(
            "BASE設定を選べず"
        )

        continue


    print(
        "BASE設定:",
        best_base
    )


    # ========================================================
    # 37. Quality閾値だけを選ぶ
    # ========================================================

    best_quality_score = -999

    best_quality_t = None


    for quality_t in QUALITY_PROB_LIST:


        signals = make_signals(

            val_p_move,

            val_p_up,

            val_p_down,

            best_base[
                "move_t"
            ],

            best_base[
                "direction_t"
            ],

            quality_prob=
                val_quality_prob,

            quality_t=
                quality_t
        )


        result = run_backtest(

            validation,

            signals,

            best_base[
                "tp"
            ],

            best_base[
                "sl"
            ]
        )


        r = (
            result[
                "returns"
            ]
        )


        if len(r) < MIN_VALIDATION_TRADES:

            continue


        score = (

            r.mean()

            * np.sqrt(
                len(r)
            )
        )


        if score > best_quality_score:


            best_quality_score = (
                score
            )


            best_quality_t = (
                quality_t
            )


    # Quality候補がなければ
    # Filterを実質無効にする
    if best_quality_t is None:

        best_quality_t = 0.0


    print(
        "Quality閾値:",
        best_quality_t
    )


    # ========================================================
    # 38. 最終Quality AI
    #
    # train_full全体から
    # OOF Qualityデータを再作成
    # ========================================================

    final_quality_oof = (

        create_quality_oof_dataset(
            train_full
        )
    )


    if final_quality_oof is None:

        continue


    final_quality_bundle = (

        fit_quality_model(
            final_quality_oof
        )
    )


    if final_quality_bundle is None:

        continue


    # ========================================================
    # 39. Base AIをtrain_fullで再学習
    # ========================================================

    final_base_models = (

        fit_base_models(

            train_full,

            trees=400
        )
    )


    if final_base_models is None:

        continue


    # ========================================================
    # 40. 完全未知Test予測
    # ========================================================

    test_p_move, \
    test_p_up, \
    test_p_down = (

        predict_base_models(

            final_base_models,

            test
        )
    )


    test_quality_prob = (

        predict_quality(

            final_quality_bundle,

            test,

            test_p_move,

            test_p_up,

            test_p_down
        )
    )


    # ========================================================
    # 41. BASE Test
    # ========================================================

    base_signals = (

        make_signals(

            test_p_move,

            test_p_up,

            test_p_down,

            best_base[
                "move_t"
            ],

            best_base[
                "direction_t"
            ]
        )
    )


    base_result = (

        run_backtest(

            test,

            base_signals,

            best_base[
                "tp"
            ],

            best_base[
                "sl"
            ]
        )
    )


    # ========================================================
    # 42. QUALITY Test
    # ========================================================

    quality_signals = (

        make_signals(

            test_p_move,

            test_p_up,

            test_p_down,

            best_base[
                "move_t"
            ],

            best_base[
                "direction_t"
            ],

            quality_prob=
                test_quality_prob,

            quality_t=
                best_quality_t
        )
    )


    quality_result = (

        run_backtest(

            test,

            quality_signals,

            best_base[
                "tp"
            ],

            best_base[
                "sl"
            ]
        )
    )


    base_stats = (

        strategy_stats(

            base_result[
                "returns"
            ]
        )
    )


    quality_stats = (

        strategy_stats(

            quality_result[
                "returns"
            ]
        )
    )


    print(
        "BASE:",
        base_stats
    )


    print(
        "QUALITY:",
        quality_stats
    )


    # ========================================================
    # 43. Fold保存
    # ========================================================

    fold_results.append({

        "fold":
            fold + 1,

        "quality_threshold":
            best_quality_t,

        "base_trades":
            base_stats[
                "trades"
            ],

        "base_win_rate":
            base_stats[
                "win_rate"
            ],

        "base_avg_return":
            base_stats[
                "avg_return"
            ],

        "base_pf":
            base_stats[
                "profit_factor"
            ],

        "base_max_dd":
            base_stats[
                "max_dd"
            ],

        "quality_trades":
            quality_stats[
                "trades"
            ],

        "quality_win_rate":
            quality_stats[
                "win_rate"
            ],

        "quality_avg_return":
            quality_stats[
                "avg_return"
            ],

        "quality_pf":
            quality_stats[
                "profit_factor"
            ],

        "quality_max_dd":
            quality_stats[
                "max_dd"
            ]
    })


    all_base_returns.extend(

        base_result[
            "returns"
        ]
    )


    all_quality_returns.extend(

        quality_result[
            "returns"
        ]
    )


    all_base_times.extend(

        base_result[
            "times"
        ]
    )


    all_quality_times.extend(

        quality_result[
            "times"
        ]
    )


# ============================================================
# 44. Fold結果
# ============================================================

fold_df = (

    pd.DataFrame(
        fold_results
    )
)


percentage_columns = [

    "base_win_rate",
    "base_avg_return",
    "base_max_dd",

    "quality_win_rate",
    "quality_avg_return",
    "quality_max_dd"
]


for col in percentage_columns:


    if col in fold_df.columns:


        fold_df[
            col
        ] *= 100


print(
    "\n\n======================================"
)

print(
    "BASE vs QUALITY Fold比較"
)

print(
    "======================================"
)


print(
    fold_df
)


# ============================================================
# 45. 総合結果
# ============================================================

base_returns = np.array(

    all_base_returns
)


quality_returns = np.array(

    all_quality_returns
)


base_total = (

    strategy_stats(
        base_returns
    )
)


quality_total = (

    strategy_stats(
        quality_returns
    )
)


final_df = pd.DataFrame(

    [

        {

            "strategy":
                "BASE",

            "trades":
                base_total[
                    "trades"
                ],

            "win_rate":
                base_total[
                    "win_rate"
                ]
                * 100,

            "average_return":
                base_total[
                    "avg_return"
                ]
                * 100,

            "profit_factor":
                base_total[
                    "profit_factor"
                ],

            "max_drawdown":
                base_total[
                    "max_dd"
                ]
                * 100
        },

        {

            "strategy":
                "QUALITY FILTER",

            "trades":
                quality_total[
                    "trades"
                ],

            "win_rate":
                quality_total[
                    "win_rate"
                ]
                * 100,

            "average_return":
                quality_total[
                    "avg_return"
                ]
                * 100,

            "profit_factor":
                quality_total[
                    "profit_factor"
                ],

            "max_drawdown":
                quality_total[
                    "max_dd"
                ]
                * 100
        }

    ]
)


print(
    "\n\n======================================"
)

print(
    "最終結果"
)

print(
    "======================================"
)


print(
    final_df
)


# ============================================================
# 46. 取引削減率
# ============================================================

if (

    base_total[
        "trades"
    ] > 0

):


    reduction = (

        1

        - quality_total[
            "trades"
        ]

        / base_total[
            "trades"
        ]

    ) * 100


    print(

        "\nQuality AIによる取引削減率:",

        round(
            reduction,
            2
        ),

        "%"
    )


# ============================================================
# 47. Equity比較
# ============================================================

if (

    len(base_returns) > 0

    and

    len(quality_returns) > 0

):


    base_equity = (

        1

        + pd.Series(
            base_returns
        )

    ).cumprod()


    quality_equity = (

        1

        + pd.Series(
            quality_returns
        )

    ).cumprod()


    plt.figure(
        figsize=(13, 6)
    )


    plt.plot(

        base_equity.values,

        label="BASE"
    )


    plt.plot(

        quality_equity.values,

        label="QUALITY FILTER"
    )


    plt.xlabel(
        "Trade Number"
    )


    plt.ylabel(
        "Growth of 1"
    )


    plt.title(
        "Base vs Trade Quality AI"
    )


    plt.legend()

    plt.grid()

    plt.show()


# ============================================================
# 48. Quality特徴量重要度
# ============================================================

if 'final_quality_bundle' in locals():


    quality_model, \
    quality_feature_names = (

        final_quality_bundle
    )


    quality_importance = (

        pd.Series(

            quality_model
            .feature_importances_,

            index=
                quality_feature_names
        )

        .sort_values(
            ascending=False
        )
    )


    print(
        "\n======================================"
    )

    print(
        "Quality AI 特徴量重要度"
    )

    print(
        "======================================"
    )


    print(
        quality_importance.head(
            20
        )
    )